# Day 3: 14 - Organization-Wide Skill Gap

Aggregated missing skills across the entire workforce.


In [ ]:
import pandas as pd

emp_df = pd.read_csv("../data/processed/employees.csv")
role_df = pd.read_csv("../data/processed/role_skills.csv")
skills_df = pd.read_csv("../data/processed/employee_skills.csv")

role_map = {r["JobRole"]: [s.strip() for s in r["RequiredSkills"].split(",")] for _, r in role_df.iterrows()}
emp_skills_map = skills_df.groupby("EmployeeID")["Skill"].apply(lambda s: set(s.str.lower())).to_dict()

gap_counts = {}
for _, emp in emp_df.iterrows():
    eid = emp["EmployeeID"]
    req = role_map.get(emp["JobRole"], [])
    has = emp_skills_map.get(eid, set())
    for s in req:
        if s.lower() not in has:
            gap_counts[s] = gap_counts.get(s, 0) + 1

gap_df = pd.DataFrame(list(gap_counts.items()), columns=["Skill", "EmployeesMissing"]).sort_values(by="EmployeesMissing", ascending=False)
gap_df["Severity"] = gap_df["EmployeesMissing"].apply(lambda x: "HIGH" if x > 200 else ("MEDIUM" if x > 100 else "LOW"))
gap_df.head(10)
